## Semantic chunking benefit for RAG application
So far, not very interesting. Need larger docs and a smarter semantic chunking strategy.

In [8]:
# --- Config ---
# Set OPENAI_API_KEY in your environment.
import dotenv, os
ENV_PATH = "/Users/douglasdaly/Documents/GitHub/Generative-AI/.env" 
dotenv.load_dotenv(ENV_PATH)

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    MarkdownHeaderTextSplitter
)
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore
import langchain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.documents import Document

from pathlib import Path
ROOT = Path("/Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks")
HOME = ROOT.parent / "assets" / "semantic_chunking"
from support import tree_markdown
display(tree_markdown(HOME))

```semantic_chunking/
└── data
    └── toyco
        ├── board_update_fy25_strategy.md
        ├── kpi_2025_04_email.txt
        ├── postmortem_smb_churn_q1_fy25.md
        ├── qbr_q1_fy25.md
        └── qbr_q3_fy24.md
```

### Load data docs

In [9]:
DATA_DIR = HOME / "data" / "toyco"

def load_toyco_docs():
    docs = []
    for path in DATA_DIR.glob("*"):
        if path.suffix.lower() not in {".md", ".txt"}:
            continue
        text = path.read_text(encoding="utf-8")
        docs.append(
            Document(
                page_content=text,
                metadata={"source": path.name},
            )
        )
    return docs

docs = load_toyco_docs()

print(len(docs), [d.metadata['source'] for d in docs])


5 ['kpi_2025_04_email.txt', 'postmortem_smb_churn_q1_fy25.md', 'qbr_q3_fy24.md', 'board_update_fy25_strategy.md', 'qbr_q1_fy25.md']


### Build naive chunker as baseline

In [10]:
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)
naive_chunks = naive_splitter.split_documents(docs)

print(len(naive_chunks), naive_chunks[0].metadata)



10 {'source': 'kpi_2025_04_email.txt'}


### Build simple semantic chunker (heading aware)

In [11]:
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]
)

md_docs = [d for d in docs if d.metadata["source"].endswith(".md")]
txt_docs = [d for d in docs if d.metadata["source"].endswith(".txt")]

# Step 1: respect headings for markdown docs
header_docs = []
for d in md_docs:
    # This returns a list[Document]
    splits = header_splitter.split_text(d.page_content)
    for sd in splits:
        # Keep original source metadata + any header metadata
        sd.metadata = {**d.metadata, **sd.metadata}
    header_docs.extend(splits)

# Step 2: keep sections reasonably sized
semantic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

semantic_md_chunks = semantic_splitter.split_documents(header_docs)
semantic_txt_chunks = semantic_splitter.split_documents(txt_docs)

semantic_chunks = semantic_md_chunks + semantic_txt_chunks
print("Semantic chunks:", len(semantic_chunks))


Semantic chunks: 24


### Embeddings, Vector Stores and Retrievers for both chunking types

In [22]:
# 1. Embeddings model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # or 3-large if you want max quality
)

# 2. Build separate vector stores
naive_store = InMemoryVectorStore.from_documents(
    naive_chunks, embedding=embeddings
)

semantic_store = InMemoryVectorStore.from_documents(
    semantic_chunks, embedding=embeddings
)

# 3. Retrievers
naive_retriever = naive_store.as_retriever(search_kwargs={"k": 3})
semantic_retriever = semantic_store.as_retriever(search_kwargs={"k": 6})

### Plug in LCEL chain

In [23]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # or whatever you’re standardizing on
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_template("""
You are an analyst for ToyCo.

Use ONLY the context to answer the question.

Question:
{question}

Context:
{context}

Answer in 2–4 short paragraphs and clearly separate:
- Business-as-usual patterns
- Genuinely unusual developments
""".strip())

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

def make_chain(retriever):
    return (
        {
            "question": RunnablePassthrough(),
            "context": retriever | format_docs,
        }
        | prompt
        | llm
    )

naive_chain = make_chain(naive_retriever)
semantic_chain = make_chain(semantic_retriever)

### Perform basic A/B test

In [24]:
# Simple one - both models answer fine. It just grabs "Long-term trends" and "Recent Anomalies" in board memo
q = (
    "Over the last two years, what is business-as-usual vs genuinely unusual "
    "in ToyCo's recent performance, especially around SMB and the Online channel?"
)

# Tougher:
q = ("For the Q1 FY25 SMB churn spike, what did ToyCo conclude were the root causes,"
     " what did they consider probably just noise, and what specific next steps did they propose?"
)

naive_answer = naive_chain.invoke(q)
semantic_answer = semantic_chain.invoke(q)


In [25]:
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))

printmd(naive_answer.content)

**Root Causes ToyCo Concluded for the Q1 FY25 SMB Churn Spike:**

ToyCo identified that the churn spike was primarily driven by price-sensitive SMB customers reacting negatively to the April price increase on Online self-serve plans. Specifically, month-to-month small-business accounts and customers acquired through heavy promotional discounts 6–12 months prior were most affected. The removal of legacy discounts and higher list prices led to cancellations concentrated in lower product tiers. Additionally, the messaging around the new pricing emphasized a "premium" positioning but failed to clearly communicate the value proposition for smaller accounts. Finally, ToyCo did not proactively notify legacy promo cohorts about the price changes, making the increase feel abrupt to those customers.

**What ToyCo Considered Probably Just Noise:**

ToyCo viewed churn fluctuations outside the identified cohorts—such as churn in Retail and Enterprise segments—as normal business-as-usual patterns, since those segments remained within typical churn ranges during the period. The short-term uplift in Online revenue following the price increase was also considered a temporary anomaly rather than a sustainable trend. Overall, the elevated churn was seen as a deviation tied specifically to the pricing change and affected cohorts, rather than a systemic or broad-based issue.

**Specific Next Steps Proposed:**

ToyCo proposed several targeted next steps to address the root causes:

- Improve communication and messaging around pricing changes, especially emphasizing value for smaller SMB accounts.
- Proactively notify legacy promo cohorts ahead of future price adjustments to reduce the shock of sudden increases.
- Reassess pricing tiers and discount strategies for month-to-month and promo-heavy SMB customers to better align with their price sensitivity.
- Monitor churn trends closely in affected cohorts to evaluate the effectiveness of these interventions and adjust as needed.

In [26]:
printmd(semantic_answer.content)

**Business-as-usual patterns:**  
Prior to Q1 FY25, SMB churn rates had remained relatively stable and within historical ranges, with slight improvements due to enhanced onboarding and success playbooks. No significant pricing changes or churn spikes were observed in Q3 FY24, indicating a steady baseline for SMB retention. The typical churn for SMB customers hovered around 13%, consistent with past performance.

**Genuinely unusual developments:**  
The Q1 FY25 SMB churn spike was directly linked to the April price increase on SMB Online self-serve plans. This caused churn in that cohort to briefly exceed 20% annualized, particularly among month-to-month accounts and promo-heavy acquisition cohorts. The spike was accompanied by increased downgrade activity and cancellations from price-sensitive small businesses, especially those acquired through paid performance marketing. While this led to a short-term revenue uplift, it raised concerns about the long-term LTV/CAC balance for these customers.

**Root causes and noise considerations:**  
ToyCo concluded that the price increase was the primary root cause of the churn spike, particularly impacting vulnerable segments like month-to-month and promo-heavy cohorts. However, they considered some of the churn elevation as potentially temporary noise—a one-time reset rather than a new baseline—since early July data showed churn trending down but not yet normalized. They are monitoring whether SMB churn will stabilize by Q3 FY25.

**Next steps proposed:**  
ToyCo plans to closely track SMB churn trends through Q3 FY25 to determine if the elevated churn persists. If it remains high, they will revisit the FY25 pricing strategy and consider implementing more gradual price changes. Additionally, they are analyzing LTV/CAC for SMB cohorts under different pricing regimes and examining the mix between Online and Retail small-business customers to better understand and address the underlying dynamics.

### New attempt
#### Download Reddit Threads for RAG

In [9]:
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Tuple
import requests

def to_json_url(url: str) -> str:
    url = url.strip()
    if url.endswith("/"):
        url = url[:-1]
    if not url.endswith(".json"):
        url = url + ".json"
    # helpful sometimes:
    if "?" not in url:
        url = url + "?raw_json=1"
    return url

def clean(text: str) -> str:
    # remove excessive whitespace
    text = re.sub(r"\s+", " ", text or "").strip()
    return text

def extract_comments(node, acc, min_score=1, depth=0, only_top_level=False):
    kind = node.get("kind")
    data = node.get("data", {})

    if kind == "t1":  # comment
        score = data.get("score", 0)
        body = clean(data.get("body", ""))

        if body and score >= min_score:
            if (not only_top_level) or (depth == 0):
                acc.append(body)

        replies = data.get("replies")
        if isinstance(replies, dict):
            children = replies.get("data", {}).get("children", [])
            for child in children:
                extract_comments(
                    child,
                    acc,
                    min_score=min_score,
                    depth=depth + 1,
                    only_top_level=only_top_level,
                )

    elif kind in {"Listing"}:
        children = data.get("children", [])
        for child in children:
            extract_comments(
                child,
                acc,
                min_score=min_score,
                depth=depth,
                only_top_level=only_top_level,
            )



def extract_comments_with_parent(node, acc, min_score=1, parent_text="", depth=0):
    kind = node.get("kind")
    data = node.get("data", {})

    if kind == "t1":
        score = data.get("score", 0)
        body = clean(data.get("body", ""))

        if body and score >= min_score:
            parent_snip = parent_text[:220]
            acc.append({
                "depth": depth,
                "score": score,
                "parent": parent_snip,
                "body": body,
            })

        replies = data.get("replies")
        if isinstance(replies, dict):
            children = replies.get("data", {}).get("children", [])
            for child in children:
                extract_comments_with_parent(
                    child, acc,
                    min_score=min_score,
                    parent_text=body,
                    depth=depth + 1
                )

    elif kind == "Listing":
        for child in data.get("children", []):
            extract_comments_with_parent(child, acc, min_score=min_score)



def download_thread(url: str, min_score=1, only_top_level=False, include_parent=False):
    json_url = to_json_url(url)
    headers = {"User-Agent": "eli5-reranker-demo/0.1 (personal research)"}
    resp = requests.get(json_url, headers=headers, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    post_listing = payload[0]
    comments_listing = payload[1]

    post_children = post_listing.get("data", {}).get("children", [])
    title = "Reddit Thread"
    if post_children:
        title = post_children[0].get("data", {}).get("title", title)

    comments = []
    for child in comments_listing.get("data", {}).get("children", []):
        if include_parent:
            extract_comments_with_parent(
                child, comments, min_score=min_score, parent_text="", depth=0
            )
        else:
            extract_comments(
                child, comments,
                min_score=min_score, depth=0, only_top_level=only_top_level
            )
    return title, comments


def write_markdown(title, comments, out_path):
    lines = []
    lines.append(f"# Thread\nTitle: {title}\n")
    lines.append("## Comments")

    if not comments:
        out_path.write_text("\n".join(lines), encoding="utf-8")
        return

    # Parent-aware dict format
    if isinstance(comments[0], dict):
        for c in comments:
            body = c.get("body", "")
            parent = c.get("parent", "")
            depth = c.get("depth", 0)
            score = c.get("score", 0)

            if depth == 0:
                lines.append(f"- ({score}) {body}")
            else:
                lines.append(f"- ({score}) Reply to: {parent} | {body}")

    # Simple string format
    else:
        for c in comments:
            lines.append(f"- {c}")

    out_path.write_text("\n".join(lines), encoding="utf-8")



def save_thread(url, min_score=1, only_top_level=False, include_parent=False) -> None:
    title, comments = download_thread(url, min_score=min_score, only_top_level=only_top_level, include_parent=include_parent)

    safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", title).strip("_")
    out_dir = Path("data/reddit_manual")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{safe_name[:80]}.md"

    write_markdown(title, comments, out_path)

    print(f"Saved {len(comments)} comments to: {out_path}")


In [10]:
if 1:
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1gjkn0t/eli5_how_do_tariffs_affect_the_price_of_goods/.json", only_top_level=True)
    save_thread("https://www.reddit.com/r/AskUS/comments/1jra630/eli5_what_is_a_tariff_and_why_it_is_a_badgood/.json", only_top_level=True)
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1mtj95m/eli5_where_does_crypto_get_its_value/.json", only_top_level=True)
    save_thread("https://www.reddit.com/r/explainlikeimfive/comments/1hag4h9/eli5where_is_all_the_money_for_crypto_coming_from/.json", only_top_level=True)
    save_thread("https://www.reddit.com/r/investing/comments/ukfxps/can_someone_explain_why_buffet_thinks_bitcoin/.json", only_top_level=True)
    save_thread("https://www.reddit.com/r/stocks/comments/1if3j75/trumps_new_tariffs_how_are_you_adjusting_your/.json", only_top_level=True)

Saved 32 comments to: data/reddit_manual/ELI5_How_do_tariffs_affect_the_price_of_goods.md
Saved 15 comments to: data/reddit_manual/ELI5_what_is_a_tariff_and_why_it_is_a_bad_good_thing.md
Saved 23 comments to: data/reddit_manual/ELI5_Where_does_crypto_get_its_value.md
Saved 46 comments to: data/reddit_manual/ELI5_Where_is_all_the_money_for_Crypto_coming_from.md
Saved 33 comments to: data/reddit_manual/Can_someone_explain_why_buffet_thinks_Bitcoin_will_go_to_zero.md
Saved 60 comments to: data/reddit_manual/Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md


In [25]:
import torch
# Globals
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
device = "mps" if torch.backends.mps.is_available() else "cpu"


In [26]:
from pathlib import Path
import re
from langchain_core.documents import Document

REDDIT_DIR = Path("data/reddit_manual")


def is_bad_comment(body: str) -> bool:
    b = (body or "").strip().lower()
    return b in {"[removed]", "[deleted]"} or len(b) < 8

def load_reddit_comment_docs(directory=REDDIT_DIR, include_post_snippet=True):
    docs = []
    for path in sorted(directory.glob("*.md")):
        text = path.read_text(encoding="utf-8")

        # Title
        title_match = re.search(r"^Title:\s*(.+)$", text, flags=re.MULTILINE)
        title = title_match.group(1).strip() if title_match else path.stem

        # Post block (optional)
        post_text = ""
        if include_post_snippet:
            post_match = re.search(
                r"^##\s+Post\s*$([\s\S]*?)(?=^##\s+Comments\s*$|$)",
                text,
                flags=re.MULTILINE,
            )
            if post_match:
                post_text = post_match.group(1).strip()

        # Comments section
        parts = re.split(r"^##\s+Comments\s*$", text, flags=re.MULTILINE)
        if len(parts) < 2:
            continue
        comments_blob = parts[1]

        bullets = re.findall(r"^\s*-\s+(.*\S.*)$", comments_blob, flags=re.MULTILINE)

        # Small, consistent context prefix
        post_snippet = ""
        if post_text:
            post_snippet = post_text[:280].replace("\n", " ")

        for i, b in enumerate(bullets):
            comment = b.strip()
            if not comment:
                continue
            if is_bad_comment(comment):
                continue
            context_prefix = f"Thread: {title}\n"
            if post_snippet:
                context_prefix += f"Post snippet: {post_snippet}\n"

            docs.append(
                Document(
                    page_content=context_prefix + f"Comment: {comment}",
                    metadata={
                        "source": path.name,
                        "thread_title": title,
                        "comment_idx": i,
                    },
                )
            )
    return docs

docs = load_reddit_comment_docs()
print("Loaded comment docs:", len(docs))


Loaded comment docs: 202


### Set up vector store and baseline retriever

In [27]:
from langchain_openai import OpenAIEmbeddings
import openai
from langchain_core.vectorstores import InMemoryVectorStore
import os
import dotenv
dotenv.load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 10})


#### LLM hook

In [28]:
from langchain_ollama import ChatOllama

# You’d need an Ollama model name you’ve pulled locally.
# Example name might differ in your setup.
llm = ChatOllama(model="mistral", temperature=0)


#### LLM Reranker

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

TOPIC_DEFS = {
    "tariffs": "A tariff is a tax on imported goods, typically paid by the importer.",
    "crypto_value": "Crypto value is market-driven and not backed by a claim on cash flows in the way many traditional assets are.",
    "fed_rates": "The federal funds rate is the interest rate on overnight lending of reserves between banks to meet short-term reserve needs; the Federal Reserve targets this rate.",
    "gold_standard": "A gold standard links a country's currency to a fixed quantity of gold, limiting discretionary expansion of the money supply.",
}

SYSTEM_RANKER_PROMPT = """
You are a careful, neutral evaluator of short-form public comments about economics and finance.
Your job is to rank how useful a comment is for answering a question.

You may use concise definition anchors provided to you to detect comments that deny or muddle basic terminology.
These anchors are not the full answer. They are only used to score definitional correctness.

Prioritize:
- correctness of the core definition
- clear mechanism explained in steps
- simple examples
- balanced caveats/tradeoffs

Avoid rewarding:
- absolutist rhetoric
- partisan cheering
- confident but vague claims
- off-topic dunking

Follow the scoring rules and return only the required JSON.
Use the full range. Most comments should fall between 4 and 8.
""".strip()


rank_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RANKER_PROMPT),
    ("human", """
User question:
{question}

Thread title:
{thread_title}

Definition anchor:
{core_definition}

Comment:
{comment}

Return JSON exactly like:
{{"score": 7}}

Scoring rules:
- If the comment denies or muddies the definition anchor, score it 0–3.
- High scores (9–10) require:
  1) Correct definition,
  2) Clear mechanism in steps,
  3) A simple example or consequence,
  4) At least one caveat/tradeoff.
- Avoid rewarding absolutist rhetoric.
Use the full range. Most comments should fall between 4 and 8.

Return ONLY the JSON object.
""".strip())
])


import json

def score_one(question, doc, core_definition):
    msg = rank_prompt.invoke({
        "question": question,
        "thread_title": doc.metadata.get("thread_title", ""),
        "core_definition": core_definition,
        "comment": doc.page_content,
    })
    out = llm.invoke(msg).content

    try:
        score = int(json.loads(out).get("score", 0))
    except Exception:
        score = 0

    return max(0, min(10, score))


def rerank(question, candidates, core_definition):
    scored = []
    for d in candidates:
        s = score_one(question, d, core_definition)
        scored.append((s, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored




#### Create screenshot ready output

In [30]:
def show_ranked(label, items, limit=None):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    if items and isinstance(items[0], tuple):
        # (score, doc)
        rows = items[:limit] if limit else items
        for i, (score, d) in enumerate(rows, start=1):
            text = d.page_content.replace("\n", " ")
            text = (text[:180] + "…") if len(text) > 180 else text
            print(f"{i:02d}. [{score:>2}/10] {d.metadata['source']} | {text}")
    else:
        # docs
        rows = items[:limit] if limit else items
        for i, d in enumerate(rows, start=1):
            text = d.page_content.replace("\n", " ")
            text = (text[:180] + "…") if len(text) > 180 else text
            print(f"{i:02d}. {d.metadata['source']} | {text}")


#### The LCEL “compare retrieval” mini-chain

In [31]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

def retrieve_candidates(question):
    return retriever.invoke(question)

compare_chain = RunnableParallel(
    question=RunnablePassthrough(),
    candidates=RunnableLambda(retrieve_candidates),
)

def run_compare(question, topic):
    pack = compare_chain.invoke(question)

    candidates = pack["candidates"]
    show_ranked("Embedding Top-50", candidates, limit=50)

    core_definition = TOPIC_DEFS[topic]
    reranked = rerank(pack["question"], candidates, core_definition)

    show_ranked("Reranked Top-15", reranked, limit=15)

    return candidates, reranked


### Run

In [32]:
# topic types = ["tariffs", "crypto_value", "fed_rates", "gold_standard"]
q = "How do tariffs affect the price of goods? Explain the mechanism and tradeoffs."
run_compare(q, "tariffs")




Embedding Top-50
01. ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: A tariff is just an import tax. The idea behind it is that another country that has lesser labor laws or big subsid…
02. ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: A tariff is a tax placed on imported or exported goods. If a tariff is placed on goods coming into a country the im…
03. ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: You want to bring your widgets and sell them in my country, because you can make them cheaper (could be for many re…
04. ELI5_How_do_tariffs_affect_the_price_of_goods.md | Thread: ELI5: How do tariffs affect the price of goods? Comment: A tariff is usually used to encourage the sale of locally produced goods. If imported widgets are cheaper than dome…
05. ELI5_How_do_tariffs_affect_the

([Document(id='61b7050f-fa99-4b09-b214-b817cc06c3c4', metadata={'source': 'ELI5_How_do_tariffs_affect_the_price_of_goods.md', 'thread_title': 'ELI5: How do tariffs affect the price of goods?', 'comment_idx': 8}, page_content="Thread: ELI5: How do tariffs affect the price of goods?\nComment: A tariff is just an import tax. The idea behind it is that another country that has lesser labor laws or big subsidies in an industry might try and undercut your local industry in your local market. Tariffs on those imported goods are used to keep your local industry competitive locally. But if there's already a local economy for the imported goods (like, for example, PC parts that are largely made in Asia), then all a tariff does is impose an extra tax on the product. The one who ultimately loses is the consumer buying the product, because the seller will pass the tax on. Even in the best case scenario where the seller absorbs *some* of the cost (because, for example, 100 sales at $50 profit are be

In [34]:
q = "Does crypto provide better risk-adjusted returns than equities? Explain simply."
run_compare(q, "crypto_value")



Embedding Top-50
01. ELI5_Where_does_crypto_get_its_value.md | Thread: ELI5: Where does crypto get its value? Comment: It has no intrinsic value. A stock returns cash to its shareholders and the underlying companies produce and sell goods and …
02. ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: Even at the current levels, crypto is a tiny asset class compared to things like equities, gold, bonds, etc
03. ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: A lot of it has to do with interest rates. When money becomes cheaper to borrow, more of it tends to find its w…
04. ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: Markets are not a zero-sum game so it is possible that more people win than lose. Money comes from investors ho…
05. ELI5_Where_is_all_the_money_f

([Document(id='6a294c93-c1e7-4859-b664-d5986a4c6867', metadata={'source': 'ELI5_Where_does_crypto_get_its_value.md', 'thread_title': 'ELI5: Where does crypto get its value?', 'comment_idx': 12}, page_content="Thread: ELI5: Where does crypto get its value?\nComment: It has no intrinsic value. A stock returns cash to its shareholders and the underlying companies produce and sell goods and services and might have other revenue streams in order to generate those cash revenues that they could partially redistribute to shareholders in the form of dividend payments, or not. Whereas crypto's value is rooted in the laws of supply and demand. Obviously share prices for companies can fluctuate day to day too, but you could easily hold a stock or a series of stocks for the long term, whereas you couldn't really do this with cryptocurrency."),
  Document(id='dff8874d-6fa7-4b99-b5fb-7d3e7ea2a977', metadata={'source': 'ELI5_Where_is_all_the_money_for_Crypto_coming_from.md', 'thread_title': 'ELI5:Wher

In [35]:
q = "When should the Fed raise or lower interest rates? Focus on mechanisms."
run_compare(q, "fed_rates")



Embedding Top-50
01. ELI5_Where_is_all_the_money_for_Crypto_coming_from.md | Thread: ELI5:Where is all the money for Crypto coming from? Comment: A lot of it has to do with interest rates. When money becomes cheaper to borrow, more of it tends to find its w…
02. Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: This is why I dollar cost average. Benjamin Graham in The Intelligent Investor talks about similar ca…
03. Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: Why adjust anything? This market has been so irrational and artificially propped up for the last few …
04. Trump_s_New_Tariffs_How_Are_You_Adjusting_Your_Investments.md | Thread: Trump’s New Tariffs – How Are You Adjusting Your Investments? Comment: Problem is that there is no safe haven. Interest rates will push up because tariffs = inflati

([Document(id='bfc5bde5-1ae8-4480-bdc7-42cc80f23478', metadata={'source': 'ELI5_Where_is_all_the_money_for_Crypto_coming_from.md', 'thread_title': 'ELI5:Where is all the money for Crypto coming from?', 'comment_idx': 22}, page_content='Thread: ELI5:Where is all the money for Crypto coming from?\nComment: A lot of it has to do with interest rates. When money becomes cheaper to borrow, more of it tends to find its way into risk assets to chase bigger returns. This is also why there was that big scare over the Japanese yen earlier this year: Lot of folks were borrowing yen to invest because it was cheaper (the yen “carry trade”) had to unwind their positions in a hurry when the exchange rate shifted against them, causing a stock market panic. With US inflation now back near the 2% target, keeping rates high becomes an unnecessary recession risk, so the Fed is finally cutting and everyone is piling in again to capitalize. There is also speculation (legit or not) that the incoming US admini